# Transformer-Based Sentiment Analysis — Sample

**Notebook 2 of 4** — LA Reddit Community Analysis, GBA 6410 Group 7

> **Requires a GPU runtime.** Runtime → Change runtime type → T4 GPU.

This notebook scores the 100,000-record verification sample with two transformer models and establishes two methodological results that the full-corpus run in notebook 2b then applies.

**Models**

| Model | Role | Native output |
|---|---|---|
| `cardiffnlp/twitter-roberta-base-sentiment-latest` | Domain-matched — trained on social media text | Three classes |
| `nlptown/bert-base-multilingual-uncased-sentiment` | General-purpose contrast | Five-point star scale |

The pairing is deliberate: one model matched to the register of the corpus and one not, so that the human-coded benchmark in notebook 3 can test whether domain matching actually improves accuracy rather than simply showing that transformers differ from lexicons.

**Input:** `sample_100k_stage1.parquet` from notebook 1
**Output:** transformer labels and confidences, keyed on `comment_id`

## Setup

PyTorch ships with Colab. The `transformers` library is often present but outdated, so it is upgraded explicitly; `accelerate` is required by `transformers` for device placement.

In [1]:
# Mount Drive

from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
# torch ships with Colab. transformers may be present but is often stale.
!pip install -q --upgrade transformers accelerate

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.6/11.6 MB 111.9 MB/s eta 0:00:00


## Configuration

The shared config block is identical to notebook 1, with model identifiers and batch settings appended. The fingerprint assertion catches the case where one copy of the config has been edited and the others have not — without it, this notebook could silently score a different set of records than notebook 1 sampled.

In [3]:
# ==================================================================
#  SHARED CONFIG — identical in all three notebooks.
#  Edit here, then copy this whole cell to notebooks 02 and 03.
#  CONFIG_FINGERPRINT below will catch it if you forget.
# ==================================================================
import re, html, hashlib, unicodedata
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from tqdm.auto import tqdm

# --- Analysis constants -------------------------------------------
RANDOM_SEED    = 42
SAMPLE_N       = 100_000
POS_T, NEG_T   = 0.05, -0.05    # lexicon thresholds (plan §4.4)
CONF_THRESHOLD = 0.50           # topic confidence cut (plan §4.7)

# --- Corpus facts (verified in plan §2) ---------------------------
EXPECTED_ROWS = 498_147
EXPECTED_COLS = 13

# --- Paths --------------------------------------------------------
BASE = Path("/content/drive/MyDrive/GBA6410")
BASE.mkdir(parents=True, exist_ok=True)

PATHS = {
    # inputs
    "raw":          BASE / "comments_clean_final.csv",
    "topics":       BASE / "topic_assignments.parquet",
    # notebook 01 outputs
    "sample":       BASE / "sample_100k_stage1.parquet",
    "scored":       BASE / "corpus_scored_lexicon.parquet",
    "lex_full":     BASE / "lexicon_scores_full.parquet",
    "lex_sample":   BASE / "lexicon_scores_sample.parquet",
    # notebook 02 outputs
    "transformer":  BASE / "transformer_scores.parquet",
    # notebook 03 artifacts
    "gold_a":       BASE / "gold_set_A_random200.xlsx",
    "gold_b":       BASE / "gold_set_B_disagree50.xlsx",
}

# --- Drift guard --------------------------------------------------
# Any change to a constant above changes this hash. Notebooks 02 and 03
# assert the same value, so a stale copy fails loudly instead of
# silently scoring a different 100K than notebook 01 sampled.
CONFIG_FINGERPRINT = hashlib.md5(
    f"{RANDOM_SEED}|{SAMPLE_N}|{POS_T}|{NEG_T}|{CONF_THRESHOLD}".encode()
).hexdigest()[:8]

np.random.seed(RANDOM_SEED)
pd.set_option("display.max_colwidth", 150)
sns.set_style("whitegrid")
plt.rcParams["figure.dpi"] = 110

print(f"Config fingerprint: {CONFIG_FINGERPRINT}")
print(f"Base: {BASE}")

# --- drift guard: must match Notebook 01's printed fingerprint -----
assert CONFIG_FINGERPRINT == "ae74dc34", (
    f"Config drift: this notebook has {CONFIG_FINGERPRINT}, "
    "Notebook 01 had something else. Re-copy Cell 2."
)

# --- notebook 02 specifics -----------------------------------------
BATCH_SIZE = 64
MAX_LENGTH = 512

MODELS = {
    "roberta": "cardiffnlp/twitter-roberta-base-sentiment-latest",
    "bert":    "nlptown/bert-base-multilingual-uncased-sentiment",
}
print("Config OK.")

Config fingerprint: ae74dc34
Base: /content/drive/MyDrive/GBA6410
Config OK.


## Preflight checks

The GPU check is an assertion here rather than a warning, inverting notebook 1. Running these models on CPU would not fail; it would simply take several hours, which is a worse outcome than stopping immediately.

In [4]:
import torch, subprocess

# ---- 1. GPU is mandatory here -------------------------------------
assert torch.cuda.is_available(), (
    "No GPU. Runtime > Change runtime type > T4 GPU, then re-run."
)
print("GPU:", torch.cuda.get_device_name(0))
print("VRAM:", round(torch.cuda.get_device_properties(0).total_memory / 1e9, 1), "GB")

# ---- 2. Inputs present --------------------------------------------
assert PATHS["sample"].exists(), f"Missing: {PATHS['sample']} (run Notebook 01)"
print(f"OK: sample {PATHS['sample'].stat().st_size / 1e6:.1f} MB")

# ---- 3. Drive writable --------------------------------------------
_probe = BASE / ".write_test"
_probe.write_text("ok"); assert _probe.read_text() == "ok"; _probe.unlink()
print("OK: Drive writable")

GPU: Tesla T4
VRAM: 15.6 GB
OK: sample 50.8 MB
OK: Drive writable


## Loading the sample

Only three columns are loaded rather than the full frame. The remainder lives in notebook 1's outputs and joins back on `comment_id` later, so there is no reason to hold it in memory alongside two models.

The maximum word count is worth checking. This corpus tops out around 229 words, comfortably inside the 512-token limit, so truncation will effectively never trigger. That is unusual — most sentiment corpora lose text at this step — and it means the scores reflect complete comments.

In [5]:
sample = pd.read_parquet(PATHS["sample"], columns=["comment_id", "sent_text", "word_count"])

assert len(sample) == SAMPLE_N, f"Expected {SAMPLE_N:,}, got {len(sample):,}"
assert not sample["comment_id"].duplicated().any(), "Duplicate comment_id"
assert sample["sent_text"].notna().all(), "Null sent_text"

print(f"Loaded {len(sample):,} comments")
print(f"Word count: median {sample['word_count'].median():.0f}, "
      f"max {sample['word_count'].max():.0f}")

Loaded 100,000 comments
Word count: median 19, max 1068


## Normalising labels across checkpoints

Different checkpoints emit different label vocabularies. Mapping them onto one scheme up front means nothing downstream has to know which model produced a given row.

**The star collapse is a decision, not a detail.** The general-purpose model rates comments from one to five stars. Mapping one and two stars to negative, three to neutral, and four and five to positive is the obvious approach, but it has a structural consequence: negative and positive each draw on two source ratings while neutral draws on only one. The model is therefore biased toward producing fewer neutral labels than a natively three-class model, independently of the text.

That bias turns out to be larger than expected, and is examined directly further down.

The `LABEL_` branch is defensive. The current checkpoint emits readable class names, but an older variant of the same model emits `LABEL_0` style identifiers; without this branch, falling back to it would produce a confusing failure several cells later rather than at the point of the mismatch.

In [6]:
STAR_MAP = {1: "negative", 2: "negative", 3: "neutral", 4: "positive", 5: "positive"}

def normalize_label(raw):
    """Map any checkpoint's output label to negative/neutral/positive."""
    s = str(raw).strip().lower()

    # nlptown: "1 star", "4 stars"
    if "star" in s:
        return STAR_MAP[int(s.split()[0])]

    # cardiffnlp -latest: already "negative"/"neutral"/"positive"
    if s in {"negative", "neutral", "positive"}:
        return s

    # older cardiffnlp checkpoints: LABEL_0 / LABEL_1 / LABEL_2
    if s.startswith("label_"):
        return ["negative", "neutral", "positive"][int(s.split("_")[1])]

    raise ValueError(f"Unrecognized label: {raw!r}")

# sanity check before spending GPU time
assert normalize_label("2 stars")  == "negative"
assert normalize_label("3 stars")  == "neutral"
assert normalize_label("POSITIVE") == "positive"
assert normalize_label("LABEL_1")  == "neutral"
print("Label mapping OK")

Label mapping OK


## Scoring function

Scoring is implemented as a function over an arbitrary set of records rather than a single pass over the sample, so that re-scoring a subset later requires no new code.

**Length sorting.** Transformers pad every sequence in a batch to the length of the longest one present. In random order, a single long comment forces its entire batch to that width and most of the computation goes into padding tokens. Sorting by length groups similar comments together so batches stay tight, which is typically a two- to threefold speedup at no cost. The `order` array records the original positions so the sort can be undone.

**Index restoration.** After sorting, result *k* corresponds to input `order[k]`, not input *k*. Writing results back through `order` puts every score on its correct record. Getting this wrong would produce a fully populated but entirely scrambled column that no assertion would catch — which is the reason every output in this workflow is keyed on `comment_id` rather than on position.

**Memory management.** The pipeline is deleted and the GPU cache cleared before the next model loads. Two models plus their activations do not fit comfortably in a T4's memory at once.

In [7]:
from transformers import pipeline
from tqdm.auto import tqdm

def score_transformer(texts, model_name, batch_size=BATCH_SIZE, desc=None):
    """Score any list of texts with any HF sentiment checkpoint.

    Returns (labels, confidences) as lists aligned to the input order.
    Sorts by length internally for batching efficiency, then restores
    the caller's order before returning.
    """
    pipe = pipeline(
        "sentiment-analysis",
        model=model_name,
        device=0,
        torch_dtype=torch.float16,   # ~2x faster on T4, negligible effect on argmax
    )

    n = len(texts)

    # Sort by length so each batch pads to a similar width.
    order  = np.argsort([len(t) for t in texts])
    sorted_texts = [texts[i] for i in order]

    raw = []
    for i in tqdm(range(0, n, batch_size), desc=desc or model_name.split("/")[-1]):
        chunk = sorted_texts[i:i + batch_size]
        raw.extend(pipe(chunk, truncation=True, max_length=MAX_LENGTH))

    # Restore the caller's original ordering.
    labels = [None] * n
    confs  = [None] * n
    for position, original_index in enumerate(order):
        labels[original_index] = normalize_label(raw[position]["label"])
        confs[original_index]  = raw[position]["score"]

    del pipe
    torch.cuda.empty_cache()
    return labels, confs

## Scoring with Twitter-RoBERTa

The result is checkpointed immediately. If the session drops while the second model is running, the first model's work is not repeated.

In [8]:
if "roberta_label" not in sample.columns:
    labels, confs = score_transformer(
        sample["sent_text"].tolist(), MODELS["roberta"], desc="RoBERTa"
    )
    sample["roberta_label"] = labels
    sample["roberta_conf"]  = confs
    sample.to_parquet(PATHS["transformer"], index=False)   # checkpoint
    print("RoBERTa done and checkpointed.")
else:
    print("RoBERTa already scored.")

print(sample["roberta_label"].value_counts(normalize=True).round(4))

config.json:   0%|          | 0.00/929 [00:00<?, ?B/s]

[transformers] `torch_dtype` is deprecated! Use `dtype` instead!


pytorch_model.bin: reconstructing file:   0%|          |  0.00B /  501MB            

pytorch_model.bin: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

...

merges.txt:   0%|          | 0.00/456k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/239 [00:00<?, ?B/s]

RoBERTa:   0%|          | 0/1563 [00:00<?, ?it/s]

[transformers] You seem to be using the pipelines sequentially on GPU. In order to maximize efficiency please use a dataset


RoBERTa done and checkpointed.
roberta_label
negative    0.5186
neutral     0.3500
positive    0.1314
Name: proportion, dtype: float64


## Scoring with multilingual BERT

Two conversions from the five-star output are computed and both are retained.

**This cell** takes the single most likely star rating and maps it to a class. This is the obvious approach and the one tried first. Its output is kept under the name `bert_label_argmax` and used only for the sensitivity comparison below.

**The next cell** sums the star probabilities into each class before taking the largest, which is the conversion used throughout the rest of the analysis. The reasoning is in the markdown above it.

In [9]:
if "bert_label_argmax" not in sample.columns:
    labels, confs = score_transformer(
        sample["sent_text"].tolist(), MODELS["bert"], desc="mBERT (argmax)"
    )
    sample["bert_label_argmax"] = labels     # naive collapse, kept for comparison
    sample["bert_conf_argmax"]  = confs
    sample.to_parquet(PATHS["transformer"], index=False)
    print("BERT (argmax collapse) done.")
else:
    print("BERT argmax already scored.")

print(sample["bert_label_argmax"].value_counts(normalize=True).round(4))

config.json:   0%|          | 0.00/953 [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B /  669MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

tokenizer_config.json:   0%|          | 0.00/39.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/872k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

mBERT:   0%|          | 0/1563 [00:00<?, ?it/s]

BERT done and checkpointed.
bert_label
negative    0.5434
positive    0.2587
neutral     0.1980
Name: proportion, dtype: float64


## The corrected conversion

Star ratings form an ordinal scale, so the probability that a comment is negative is the probability of one star **plus** the probability of two stars. Taking the single most likely star instead allows three stars to win on a plurality while negative or positive support sits divided across two adjacent ratings that are never added together.

A comment scoring 1★ = 0.30, 2★ = 0.25, 3★ = 0.35 illustrates it. The most-likely-star approach sees three stars leading and returns neutral. Summing gives negative 0.55 against neutral 0.35 and returns negative. The second is correct.

This also makes the confidence values comparable with RoBERTa's. Under the naive conversion, the recorded confidence is a maximum over five classes with a floor of 0.20, while RoBERTa's is a maximum over three with a floor of 0.333 — different quantities that should not be placed side by side. After summing, both are maxima over three classes.

The cell requests the full probability distribution rather than the top rating, then aggregates.

In [13]:
# --- BERT: collapse five stars to three classes by SUMMING -----------
# Requests the full 5-class distribution, then sums the star
# probabilities belonging to each collapsed class. This is the
# conversion used throughout the analysis; the argmax version above is
# retained only for the sensitivity comparison.

STAR_TO_CLASS = {1: "negative", 2: "negative", 3: "neutral",
                 4: "positive", 5: "positive"}

def score_bert_collapsed(texts, batch_size=BATCH_SIZE):
    pipe = pipeline("sentiment-analysis", model=MODELS["bert"],
                    device=0, torch_dtype=torch.float16, top_k=None)

    n = len(texts)
    order = np.argsort([len(t) for t in texts])
    sorted_texts = [texts[i] for i in order]

    raw = []
    for i in tqdm(range(0, n, batch_size), desc="mBERT (collapsed)"):
        raw.extend(pipe(sorted_texts[i:i + batch_size],
                        truncation=True, max_length=MAX_LENGTH))

    labels = [None] * n
    confs  = [None] * n
    for position, original_index in enumerate(order):
        agg = {"negative": 0.0, "neutral": 0.0, "positive": 0.0}
        for d in raw[position]:                     # 5 dicts, one per star
            star = int(str(d["label"]).split()[0])
            agg[STAR_TO_CLASS[star]] += d["score"]
        best = max(agg, key=agg.get)
        labels[original_index] = best
        confs[original_index]  = agg[best]

    del pipe; torch.cuda.empty_cache()
    return labels, confs

labels, confs = score_bert_collapsed(sample["sent_text"].tolist())

# How far the two conversions diverge. Compared against the argmax
# labels from the previous cell, NOT against bert_label — which does
# not exist until the assignment below.
print("Agreement with argmax collapse:",
      (pd.Series(labels) == sample["bert_label_argmax"]).mean())

# The summed collapse is the one adopted: assign both label and
# confidence, so the two never disagree with each other on disk.
sample["bert_label"] = labels
sample["bert_conf"]  = confs
sample.to_parquet(PATHS["transformer"], index=False)

print(sample["bert_conf"].describe().round(3))

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

mBERT (collapsed):   0%|          | 0/1563 [00:00<?, ?it/s]

Agreement with argmax collapse: 0.88114
count    100000.000
mean          0.667
std           0.174
min           0.334
25%           0.517
50%           0.657
75%           0.813
max           0.998
Name: bert_conf, dtype: float64


In [15]:
# Class shares under the adopted (summed) conversion.
print(sample["bert_label"].value_counts(normalize=True).round(4))

bert_label
negative    0.6245
positive    0.2922
neutral     0.0832
Name: proportion, dtype: float64


**Result.** The two conversions disagree on **11.9%** of comments — far more than the small correction anticipated. The neutral share more than halves, from 19.8% under the naive conversion to 8.3% under the correct one:

| Conversion | Positive | Neutral | Negative |
|---|---|---|---|
| Most likely star | 25.9% | 19.8% | 54.3% |
| Summed probabilities | 29.3% | 8.3% | 62.6% |

Both figures are retained so the impact of this decision can be reported as a sensitivity comparison.

The confidence floor also moved from 0.204 to 0.334, matching RoBERTa's 0.336, confirming that both are now maxima over three classes and can be compared directly.

The substantive consequence is that BERT, under the correct conversion, is a very poor detector of neutral content. The human-coded benchmark in notebook 3 confirms this: it identifies 4 of 37 genuinely neutral comments, a recall rate of 10.8%.

## Distributions, confidence, and agreement

The figure to watch is RoBERTa's neutral share. Notebook 1 established that the lexicons' apparent neutral classes are largely abstentions — genuine neutral judgments amount to 8.5% of the corpus for TextBlob and 1.7% for VADER. RoBERTa's neutral class is trained, so whatever it reports is a real prediction rather than a gap in a dictionary.

Confidence here is a softmax over three classes, so the floor is 0.333 rather than the 0.20 seen in the LDA topic probabilities. These models are usually confident; a high proportion of low-confidence predictions would be the surprising outcome.

In [16]:
dist = pd.DataFrame({
    "RoBERTa": sample["roberta_label"].value_counts(normalize=True),
    "BERT":    sample["bert_label"].value_counts(normalize=True),
}).reindex(["positive", "neutral", "negative"]).fillna(0)

print("Label shares (%):")
print((dist * 100).round(2), "\n")

print("Confidence:")
print(sample[["roberta_conf", "bert_conf"]].describe().round(3), "\n")

for m in ["roberta", "bert"]:
    low = (sample[f"{m}_conf"] < 0.50).mean()
    print(f"{m}: {low:.1%} of predictions below 0.50 confidence")

print("\nAgreement between the two transformers: "
      f"{(sample['roberta_label'] == sample['bert_label']).mean():.1%}")
print(pd.crosstab(sample["roberta_label"], sample["bert_label"],
                  rownames=["RoBERTa"], colnames=["BERT"]))

Label shares (%):
          RoBERTa   BERT
positive    13.14  29.22
neutral     35.00   8.33
negative    51.86  62.45 

Confidence:
       roberta_conf   bert_conf
count    100000.000  100000.000
mean          0.740       0.667
std           0.149       0.174
min           0.336       0.334
25%           0.616       0.517
50%           0.756       0.657
75%           0.871       0.813
max           0.993       0.998 

roberta: 5.9% of predictions below 0.50 confidence
bert: 21.8% of predictions below 0.50 confidence

Agreement between the two transformers: 56.2%
BERT      negative  neutral  positive
RoBERTa                              
negative     41627     2928      7308
neutral      18807     4430     11760
positive      2020      967     10153


**Result.** RoBERTa assigns neutral to 35.0% of comments, against genuine lexicon neutrals of 8.5% and 1.7%. This is direct evidence for the central claim of the comparison: the lexicon methods cannot represent neutrality, and a trained three-class model can.

The two transformers agree on only **56.2%** of comments — barely above the 55.9% recorded between the two lexicons in notebook 1. Model family is therefore not what determines agreement.

The structure of the disagreement is informative. The two models agree well at the poles but diverge sharply in the middle: of RoBERTa's 35,000 neutral predictions, BERT labels fewer than one in eight as neutral, sending 40.6% of them to negative. That follows directly from the star-collapse asymmetry described above, and is an artifact of the label scheme rather than a genuine disagreement about sentiment.

## Sensitivity to capitalisation and emoji

Notebook 1 measured how much the text representation matters for the lexicon methods: 0.04% of TextBlob labels changed and 0.80% of VADER's. Those figures suggested a minor effect.

Transformers tokenise capitalisation and emoji as meaningful subword units rather than checking them against rules, so their sensitivity could be substantially higher. This cell runs the same controlled comparison — same records, same model, only the input text differing — on a 20,000-record subset.

In [11]:
SUB_N = 20_000
sub = sample.sample(n=SUB_N, random_state=RANDOM_SEED)[["comment_id", "roberta_label"]]

old_text = (pd.read_parquet(PATHS["sample"], columns=["comment_id", "clean_body_stop"])
              .set_index("comment_id")
              .loc[sub["comment_id"], "clean_body_stop"]
              .fillna("").tolist())

old_labels, _ = score_transformer(old_text, MODELS["roberta"], desc="RoBERTa (old col)")
sub["roberta_label_old"] = old_labels

changed = sub["roberta_label"] != sub["roberta_label_old"]
print(f"RoBERTa: {changed.mean():.2%} of labels differ ({changed.sum():,} of {SUB_N:,})")
print("  (Notebook 01: TextBlob 0.04%, VADER 0.80%)\n")
if changed.any():
    print(pd.crosstab(sub.loc[changed, "roberta_label_old"],
                      sub.loc[changed, "roberta_label"],
                      rownames=["old"], colnames=["new"]))

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

[transformers] RobertaForSequenceClassification LOAD REPORT from: cardiffnlp/twitter-roberta-base-sentiment-latest
Key                         | Status     |  | 
----------------------------+------------+--+-
roberta.pooler.dense.weight | UNEXPECTED |  | 
roberta.pooler.dense.bias   | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


RoBERTa (old col):   0%|          | 0/313 [00:00<?, ?it/s]

RoBERTa: 5.08% of labels differ (1,016 of 20,000)
  (Notebook 01: TextBlob 0.04%, VADER 0.80%)

new       negative  neutral  positive
old                                  
negative         0      425        35
neutral        228        0       223
positive        14       91         0


**Result.** RoBERTa changes on **5.08%** of labels: more than six times VADER's rate and over a hundred times TextBlob's.

For the model ultimately selected, one label in twenty depends on whether capitalisation and emoji were preserved. That reframes the text-representation decision from a minor cleaning preference into a substantive methodological point, and it is the reason the decision is documented in full rather than in a sentence.

The direction is also consistent: the largest single flow is negative to neutral, indicating that preserved signal is softening judgments the stripped text read as hostile.

In [14]:
# --- Examples of labels that flipped when case/emoji were preserved ---
texts = (pd.read_parquet(PATHS["sample"], columns=["comment_id", "sent_text"])
           .set_index("comment_id")["sent_text"])

flips = sub[changed].join(texts, on="comment_id")

# largest flow first: negative -> neutral (425 comments)
for old, new in [("negative", "neutral"), ("neutral", "positive"),
                 ("neutral", "negative")]:
    rows = flips[(flips["roberta_label_old"] == old) &
                 (flips["roberta_label"] == new)]
    print(f"\n===== {old} → {new}  ({len(rows)} comments) =====")
    for t in rows["sent_text"].head(3):
        print(f"  • {t[:150]}")


===== negative → neutral  (425 comments) =====
  • I once walked from in n out to the airport with my luggage... Never again
  • Ominous? It's just saying there'll be a lot of traffic tomorrow
  • Intentional title gore

===== neutral → positive  (223 comments) =====
  • I was down there yesterday! Going back tomorrow!
  • News flash. Company likes money. Will go where they make most money.
  • Ah shit I should posted the link—THANK YOU

===== neutral → negative  (228 comments) =====
  • Unless it's winning like Charlie Sheen 🙁
  • Have you heard of......... the homeless? 😬
  • Yeah which is why I said an average day. It regularly gets worse than that in those regions, but on an average day, sure that's totally within the nor


**Examples.** Three of the changed comments show the mechanism directly:

- *"Unless it's winning like Charlie Sheen 🙁"* — neutral on the words alone; the emoji carries the sentiment
- *"Have you heard of......... the homeless? 😬"* — the same pattern, with the emoji supplying the tone
- *"Ah shit I should posted the link—THANK YOU"* — capitalisation marks genuine gratitude, and the profanity is not hostile

Not every change is attributable to capitalisation or emoji specifically. The two text versions also differ in markdown handling, apostrophes, and dashes, so the finding is best described as the effect of the richer text representation overall.

## Saving

Both conversions of the BERT output are saved. The summed version is used throughout the analysis; the argmax version supports the sensitivity comparison reported above.

All outputs are keyed on `comment_id` so that notebook 3 can join the lexicon scores, transformer scores, and topic assignments without any of the three depending on row order.

In [12]:
out_cols = ["comment_id", "roberta_label", "roberta_conf",
            "bert_label", "bert_conf",
            "bert_label_argmax", "bert_conf_argmax"]
sample[out_cols].to_parquet(PATHS["transformer"], index=False)

print("Saved:", PATHS["transformer"])
print(sample[out_cols].head())

Saved: /content/drive/MyDrive/GBA6410/transformer_scores.parquet
  comment_id roberta_label  roberta_conf bert_label  bert_conf
0    o85x3h8      negative      0.926192   negative   0.669623
1    ot07bhy      negative      0.866974   negative   0.851960
2    ooozgpa       neutral      0.551684   negative   0.340037
3    o7iyj5v      negative      0.583532   positive   0.448153
4    obp1p4x       neutral      0.712247   positive   0.541892


---
## Outputs

| Column | Contents |
|---|---|
| `roberta_label`, `roberta_conf` | Twitter-RoBERTa, three-class output |
| `bert_label`, `bert_conf` | Multilingual BERT, summed star collapse — used throughout |
| `bert_label_argmax`, `bert_conf_argmax` | Multilingual BERT, naive star collapse — sensitivity comparison only |

Written to `transformer_scores.parquet`.

## Findings established here

1. The two conversions of the five-star output disagree on 11.9% of comments; the naive one is biased toward neutral. The summed conversion is adopted.
2. RoBERTa identifies neutral content at 35.0%, against genuine lexicon neutrals of 8.5% and 1.7%.
3. The two transformers agree on only 56.2% of comments, barely above the two lexicons' 55.9%.
4. Preserving capitalisation and emoji changes 5.08% of RoBERTa's labels, against 0.80% for VADER and 0.04% for TextBlob.

**Next:** notebook 2b applies both models to the full corpus using resumable chunked processing.